# **Miniproject 2**
## **~Large~ Small Language Model**

### **Objective**
Implement a transformer-based, character-level language model (GPT-like) and train it on the Shakespeare dataset. By the end of this project, you should be able to generate Shakespearean-like text given a seed string.

You will probably want to train the model on a GPU. You can use free GPUs on [Google Colab](https://colab.research.google.com/?utm_source=scs-index).

### **Dataset**:

The Shakespeare dataset contains the complete works of William Shakespeare, including his plays, poems, and sonnets.

[**Download link**](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt)

In a character-level language model, each character in the input data is mapped to its respective index from a dictionary. The input to the model is in the form (B, N), where B is the batch size and N is the number of tokens for each sequence. The model was tested with B=N=128, but feel free to explore different values.

An interface for the dataset class that takes care of tokenization is provided below.



```python
from torch.utils.data import Dataset

class CharDataset(Dataset):
    """
    Emits batches of characters.

    Adapted from "https://github.com/karpathy/minGPT".
    """

    def __init__(self, config, data):

        chars = ... # get characters from the input data
        self.stoi = { ch:i for i,ch in enumerate(chars) } # map characters to integer indices

        ...

    def get_vocab_size(self):
        raise NotImplementedError()

    def __len__(self):
        raise NotImplementedError()

    def __getitem__(self, idx):
        # grab a chunk of (block_size + 1) characters from the data
        # encode every character to an integer
        # return the chunk and the shifted version as tensors
        pass
```




### **Requirements**

#### **Architecture**

Implement the Transformer's decoder-only structure.
This includes

* input token embeddings
* the causal multi-head self-attention mechanism
* feed-forward neural networks
* positional encodings, residual connections, layer normalizations.

The project was tested with $12$ layers, $8$ attention heads, and $768$ embedding dimensions, on a single GPU.

The `forward` method for the entire model has the following form:

```
tok_emb = WTE(idx) # token embeddings
pos_emb = WPE(pos) # position embeddings
x = Dropout(tok_emb + pos_emb)
for Block in Blocks:
    x = Block(x)
x = Final_LayerNorm(x)
logits = LM_Head(x)
```

The `forward` method for the transformer block has the following form:



```
x = x + self.CausalSelfAttn(self.LayerNorm_1(x))
out = x + self.MLP(self.LayerNorm_2(x))
```

---

#### **Training**

In a character-level transformer language model, the goal is to predict the next character in a sequence given the previous characters. To train such a model effectively, we use two versions of our data: the input sequence and a shifted version of this sequence, which serves as the target for our predictions.

Preprocess the dataset to a character-level representation.
Use a sliding window approach for sequence chunks (e.g., window size of $128$ characters).
Implement causal masking for the self-attention mechanism.
Use the [Adam](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html) optimizer and the cross-entropy loss.

**Optional**:

* Implement a learning rate decay strategy
* Implement gradient clipping

---


#### **Evaluation and Inference**

* Monitor the cross-entropy loss. Use a seed string to initialize the model and generate Shakespearean-like text.

* In order to generate the characters, at each generation step you can either select the character with the highest probability, or you can sample according to the output distribution.

The high-level pseudocode for generation is:

```python
model.eval()
with torch.no_grad():
    context = "O God, O God!"
    tokenized_context = tokenize(context)
    # the model should implement a method to generate tokens given a prompt
    y = model.generate(tokenized, ...)
    completion = tokens_to_string(y)
```

**Optional**:
* Compute the [perplexity](https://medium.com/@priyankads/perplexity-of-language-models-41160427ed72#:~:text=Intuitively%2C%20perplexity%20means%20to%20be,loss%20obtained%20from%20the%20model.) metric for quantitative evaluation.

### **Example Outputs**

The following are my outputs after $6000$ steps of training, with the seed string "O God, O God!"



```
O God, O God! neither? unto the base very ears,
As damned with it.

DUKE OF YORK:
Away! Once more, one word.

RICHARD:
Clove, dear so; and therein my son will be
false of woe: if ye seems to be the mother
Of gracious order this time when R going kinsperse eyes,
What dost bewreck her fairer drying tears.

NORTHUMBERLAND:
Have you forgot the Duke of Norfolk, get him to
again; and and agilic: there is my spirit
So maly did must such a marble perfection.

ELBOW:
Come, bring them with oaths, and so deliver
```


### Resources:

* Vaswani et al., "Attention is All You Need": [link](https://arxiv.org/abs/1706.03762)

* Illustrated Transformer by Jay Alammar: [link](https://jalammar.github.io/illustrated-transformer/)

* OpenAI GPT-2 Paper: [link](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

* Deep Learning Course slides on transformers: [link](https://fleuret.org/dlc/materials/dlc-handout-13-3-transformers.pdf)

In [41]:
import torch
from torch.utils.data import Dataset
import math
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_
from architecture import Shakespeare

In [ ]:
class CharDataset(Dataset):
    def __init__(self, config, data, chars=None):

        self.block_size = config.block_size

        # if vocab not given, build it, but we usually pass all of them to both val and train
        if chars is None:
            chars = sorted(list(set(data)))

        self.char_to_num = {ch: i for i, ch in enumerate(chars)}
        self.num_to_char = {i: ch for ch, i in self.char_to_num.items()}
        self.vocab_size = len(chars)

        self.data_encoded = torch.tensor([self.char_to_num[c] for c in data], dtype=torch.long)
    def get_vocab_size(self):
        return self.vocab_size

    def __len__(self):
        return len(self.data_encoded) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data_encoded[idx:idx + self.block_size + 1]
        return chunk[:-1], chunk[1:]


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

block_size = 128

with open("Shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))

n = int(0.95 * len(text))
train_text = text[:n]
val_text   = text[n:]

cfg = type("cfg", (), {"block_size": block_size})()
train_dataset = CharDataset(cfg, train_text, chars=chars) # Datasets
val_dataset = CharDataset(cfg, val_text, chars=chars)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True) # Loaders
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

model = Shakespeare(
    vocab_size = train_dataset.get_vocab_size(),
    block_size = block_size,
    embed_dim = 768,
    num_heads = 8,
    n_layers = 12
).to(device)

base_lr = 0.0003
optimizer = torch.optim.Adam(model.parameters(), lr=base_lr)
total_step = 6000 # Clear overfitting beyond that

In [44]:
print(torch.cuda.is_available())

True


In [ ]:
@torch.no_grad()
def evaluate(model, val_loader): # Evaluation for validation set
    model.eval() # eval on val set
    total_loss = 0
    total_tokens = 0

    for i, (x, y) in enumerate(val_loader):
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        total_loss += loss.item() * y.numel() # number of elem as weight
        total_tokens += y.numel()

    model.train() # back to train mode
    avg_loss = total_loss / total_tokens
    return avg_loss


In [ ]:
for i, (x, y) in enumerate(train_loader) :
    x = x.to(device)
    y = y.to(device)

    optimizer.zero_grad()
    _, loss = model(x, y)
    loss.backward()
    clip_grad_norm_(model.parameters(), 1)
    optimizer.step()
    # Lr decay
    lr = base_lr * (1 - (i / total_step))

    for param_group in optimizer.param_groups :
        param_group['lr'] = lr

    if i % 100 == 0 :
        print(f"step {i}, loss = {loss.item():.4f}")

    if i % 1000 == 0 and i > 0:
        val_loss = evaluate(model, val_loader)
        print(f"step {i}, val. loss = {val_loss:.4f}")
        torch.save(model.state_dict(), f"model_step_{i}.pth")
        model.eval()
        with torch.no_grad():
            context = "O God, O God!"
            tokenized_context = torch.tensor([[train_dataset.char_to_num[c] for c in context]], device=device)
            # the model should implement a method to generate tokens given a prompt
            y = model.generate(tokenized_context, max_new_tokens=300)
            completion = "".join(train_dataset.num_to_char[i] for i in y[0].tolist())
            print(f"=== Completion at step {i} ===")
            print(completion)

    if i == total_step :
        break


step 0, loss = 4.2740
step 100, loss = 2.4236
step 200, loss = 2.0392
step 300, loss = 1.7939
step 400, loss = 1.6571
step 500, loss = 1.5439
step 600, loss = 1.4625
step 700, loss = 1.4273
step 800, loss = 1.3937
step 900, loss = 1.3498
step 1000, loss = 1.3080
step 1000, val. loss = 1.5368
=== Completion at step 1000 ===
O God, O God!
O, heaven bless!

GLOUCESTER:
Errone and tumbling of it is restraint,
Give, the enemies, let the earth of blind
In my rocking braces upon the pable; after man
Belived their harch, shalt come from thy fair grue.
This is be so, more boding and vitiog.
On I
O, come this intemplicious: sir I did sight.
step 1100, loss = 1.2159
step 1200, loss = 1.1803
step 1300, loss = 1.1783
step 1400, loss = 1.0985
step 1500, loss = 1.0936
step 1600, loss = 1.0581
step 1700, loss = 1.0421
step 1800, loss = 0.9753
step 1900, loss = 0.9362
step 2000, loss = 0.9098
step 2000, val. loss = 1.7140
=== Completion at step 2000 ===
O God, O God! For I aught be gone that grave!
For

KeyboardInterrupt: 

In [ ]:
# save model
torch.save(model.state_dict(), "gpt_shakespeare.pth")

In [ ]:
model.load_state_dict(torch.load("gpt_shakespeare.pth", map_location=device))

<All keys matched successfully>

In [56]:
model.eval()
with torch.no_grad():
    context = "O God, O God!"
    tokenized_context = torch.tensor([[train_dataset.char_to_num[c] for c in context]], device=device)
    # the model should implement a method to generate tokens given a prompt
    y = model.generate(tokenized_context, max_new_tokens=500)
    completion = "".join(train_dataset.num_to_char[i] for i in y[0].tolist())

print(completion)

O God, O God! the nurse, and set the duke at Calais.

DUKE OF AUMERLE:
Some honest Christian trust me with a gage
That Norfolk lies: Tybalt comfort not to me.
God some one that was devilish paged me.

DUKE VINCENTIO:
Here comes Aufidius Mantua: what was Romeo?

PRINCE:
My gracious lord, he might be the first:
If thou art rich, thou'rt taken;
For so 'twere perpetual daughters.

GLOUCESTER:
So do not with him good while the end playets,
And by the fires of heaven, I'll leave the foe
And make my wars on your wa


In [62]:
T = train_dataset.data_encoded # All text in a tensor
# Useless to use the loader so we directly use T, it is faster

total_loss = 0
total_tokens = 0

with torch.no_grad():
    for i in range(0, len(T)-1, block_size):

        if i%100 == 0:
            print(f"Process at {i}/{len(T)}", end="\r")   

        end_idx = min(i+block_size, len(T)-1) 
        x = T[i:end_idx].reshape(1, -1).to(device)
        y = T[i+1:end_idx+1].reshape(1, -1).to(device)

        _, loss = model(x, y)

        total_loss += loss.item() * (y.shape[0] * y.shape[1]) # Item of tensor of cross entropy so nll
        total_tokens += (y.shape[0] * y.shape[1])


ppl = math.exp(total_loss / total_tokens)
print("Perplexity = ", ppl)

T2 = val_dataset.data_encoded
total_loss2 = 0
total_tokens2 = 0

with torch.no_grad():
    for i in range(0, len(T2)-1, block_size):

        if i%100 == 0:
            print(f"Process at {i}/{len(T2)}", end="\r")   

        end_idx = min(i+block_size, len(T2)-1) 
        x2 = T2[i:end_idx].reshape(1, -1).to(device)
        y2 = T2[i+1:end_idx+1].reshape(1, -1).to(device)

        _, loss2 = model(x2, y2)

        total_loss2 += loss2.item() * (y2.shape[0] * y2.shape[1])
        total_tokens2 += (y2.shape[0] * y2.shape[1])


ppl2 = math.exp(total_loss2 / total_tokens2)
print("Validation Perplexity = ", ppl2)

Perplexity =  1.1843443147242594
Validation Perplexity =  34.795042086367005
